In [ ]:
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv
!pip install -q numba cupy-cuda12x pytest pynvml
import numba.cuda as cuda
import cupy as cp
if not cuda.is_available():
    raise SystemError("CRITICAL: CUDA GPU not detected! Go to Runtime > Change runtime type > Select  T4 GPU.")
gpu = cuda.get_current_device()
print(f"\n[+] Environment Certified for Autonomous CUDA Lab")
print(f" Target Device : {gpu.name}")
print(f" Compute Capable : {gpu.compute_capability}")
print(f" CuPy Device Name : {cp.cuda.runtime.getDeviceProperties(0)['name']}")
print(f" Warp Size : {gpu.WARP_SIZE} threads")
print(f" Max Block Threads: {gpu.MAX_THREADS_PER_BLOCK}")
print(f" Max Shared/Block : {gpu.MAX_SHARED_MEMORY_PER_BLOCK / 1024:.1f} KB\n")

name, driver_version, memory.total [MiB]
Tesla T4, 580.82.07, 15360 MiB

[+] Environment Certified for Autonomous CUDA Lab
 Target Device : Tesla T4
 Compute Capable : ComputeCapability(major=7, minor=5)
 CuPy Device Name : b'Tesla T4'
 Warp Size : 32 threads
 Max Block Threads: 1024
 Max Shared/Block : 48.0 KB



In [ ]:
@cuda.jit

def  matrix_add_2d(A, B, C):
   col,row = cuda.grid(2)

   if (row < C.shape[0] and col < C.shape[1]):
     C[row,col] = A[row,col] + B[row, col];




In [ ]:
import numpy as np
import numba.cuda as cuda

@cuda.jit
def vector_add_1d(a, b, out):
    i = cuda.grid(1)
    if i < len(a):
        out[i] = a[i] + b[i]

def run_task_1(N):
    a_host = np.random.randn(N).astype(np.float32)
    b_host = np.random.randn(N).astype(np.float32)
    out_host = np.zeros(N, dtype=np.float32)

    a_device = cuda.to_device(a_host)
    b_device = cuda.to_device(b_host)
    out_device = cuda.to_device(out_host)

    threads_per_block = 256
    blocks_per_grid = (N + threads_per_block - 1) // threads_per_block

    vector_add_1d[blocks_per_grid, threads_per_block](a_device, b_device, out_device)

    out = out_device.copy_to_host()

    np.testing.assert_allclose(out, a_host + b_host, atol=1e-5)
    print(f" N = {N:<7} | Grid: {blocks_per_grid:<5} Blocks | Block: {threads_per_block} Threads")

test_lengths = [1, 255, 256, 1024, 1000003]
print("=== Running Task 01: 1D Vector Addition ===")
for length in test_lengths:
    run_task_1(length)

=== Running Task 01: 1D Vector Addition ===
 N = 1       | Grid: 1     Blocks | Block: 256 Threads
 N = 255     | Grid: 1     Blocks | Block: 256 Threads
 N = 256     | Grid: 1     Blocks | Block: 256 Threads
 N = 1024    | Grid: 4     Blocks | Block: 256 Threads
 N = 1000003 | Grid: 3907  Blocks | Block: 256 Threads


/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
/usr/local/lib/python3.13/dist-packages/numba_cuda/numba/cuda/dispatcher.py:716: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [ ]:
@cuda.jit
def rgb_to_grayscale(img_rgb, img_gray):
    x, y = cuda.grid(2)
    if x < img_rgb.shape[0] and y < img_rgb.shape[1]:
        r = img_rgb[x, y, 0]
        g = img_rgb[x, y, 1]
        b = img_rgb[x, y, 2]
        img_gray[x, y] = 0.299 * r + 0.587 * g + 0.114 * b

In [ ]:
@cuda.jit

def matrix_transpose_naive(A, B):
  A = cuda.grid(1);
  B = cuda.grid(2);


